In [5]:
import pandas as pd
from tqdm import tqdm


In [6]:
def load_subset(filepath, max_ratings=500000):
    data = []
    movie_id = None

    with open(filepath, "r", encoding="utf-8") as f:

        for line in tqdm(f):

            if line.endswith(":\n"):
                movie_id = int(line[:-2])

            else:
                user_id, rating, date = line.strip().split(",")

                data.append([
                    int(user_id),
                    movie_id,
                    int(rating),
                    date
                ])

                if len(data) >= max_ratings:
                    break

    return pd.DataFrame(
        data,
        columns=["user_id","movie_id","rating","date"]
    )

In [7]:
df = load_subset(
    "../data/combined_data_1.txt",
    max_ratings=500000
)

500147it [00:00, 715863.02it/s]


In [9]:
df = df[['user_id','movie_id','rating']]

# Item-Based Collaborative Filtering

In [10]:
user_counts = df['user_id'].value_counts()
movie_counts = df['movie_id'].value_counts()

active_users = user_counts[user_counts >= 5].index
popular_movies = movie_counts[movie_counts >= 50].index

filtered_df = df[
    (df['user_id'].isin(active_users)) &
    (df['movie_id'].isin(popular_movies))
]

print(filtered_df.shape

(167232, 3)


In [12]:
user_movie_matrix = filtered_df.pivot_table(
    index='user_id',
    columns='movie_id',
    values='rating'
)

user_movie_filled = user_movie_matrix.fillna(0)

In [13]:
from sklearn.metrics.pairwise import cosine_similarity
movie_similarity = cosine_similarity(
    user_movie_filled.T
)

In [14]:
movie_similarity_df = pd.DataFrame(
    movie_similarity,
    index=user_movie_filled.columns,
    columns=user_movie_filled.columns
)

movie_similarity_df.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,139,140,141,142,143,144,145,146,147,148
movie_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.048129,0.023496,0.066584,0.056447,0.022881,0.077977,0.036773,0.024685,0.029409,...,0.051722,0.061497,0.026578,0.015098,0.062382,0.042938,0.046027,0.043727,0.040693,0.005597
2,0.048129,1.000000,0.025200,0.078420,0.047904,0.031157,0.106879,0.026016,0.065487,0.066003,...,0.100557,0.079883,0.076048,0.041389,0.031072,0.060230,0.053138,0.024494,0.079326,0.013538
3,0.023496,0.025200,1.000000,0.030195,0.015287,0.065194,0.028007,0.091054,0.020546,0.066259,...,0.050335,0.024395,0.031644,0.030989,0.111983,0.031805,0.151713,0.023579,0.051543,0.015832
4,0.066584,0.078420,0.030195,1.000000,0.037283,0.030490,0.073602,0.023196,0.030065,0.045453,...,0.159272,0.073981,0.055749,0.057006,0.030365,0.049773,0.050218,0.023720,0.071699,0.010877
5,0.056447,0.047904,0.015287,0.037283,1.000000,0.023431,0.059538,0.022586,0.041952,0.026394,...,0.030249,0.033145,0.021523,0.029973,0.068533,0.021138,0.025791,0.026479,0.031071,0.014958


In [20]:
user_movie_matrix.shape

(23332, 148)

In [16]:
def recommend_movies(movie_id, n=5):

    similar_movies = (
        movie_similarity_df[movie_id]
        .sort_values(ascending=False)
    )

    return similar_movies.iloc[1:n+1]

In [22]:
recommend_movies(28)

movie_id
30     0.593139
58     0.500833
143    0.483614
111    0.454889
118    0.425790
Name: 28, dtype: float64